# Stage 6 — preprocessing profile v1 and F3 candidate

This notebook runs the approved FoR-only Stage 6 validation from an exact clean commit. It re-screens shortcut cues, checks resampling invariance, and measures preprocessing/GPU compute without an optimizer, training, model selection, or checkpoints. Run it only after the implementation commit is reviewed locally.

## 1. Choose transfer mode and check out the exact implementation commit

Replace `IMPLEMENTATION_COMMIT` with the commit approved for the official run. Leave `TRANSFER_MODE` as `manual` when Colab cannot mount Drive, or change it to `drive` when mounting works. The checkout must remain clean.

In [ ]:
from pathlib import Path
import os, shutil, subprocess

TRANSFER_MODE = 'manual'  # 'manual' or 'drive'
IMPLEMENTATION_COMMIT = 'REPLACE_WITH_APPROVED_COMMIT'
if TRANSFER_MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
elif TRANSFER_MODE != 'manual':
    raise ValueError("TRANSFER_MODE must be 'manual' or 'drive'")
REPO = Path('/content/voice-spoofing-detection')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', 'https://github.com/kahlanzzi/voice-spoofing-detection.git', str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', IMPLEMENTATION_COMMIT], cwd=REPO, check=True)
assert not subprocess.run(['git', 'status', '--porcelain'], cwd=REPO, check=True, capture_output=True, text=True).stdout
os.chdir(REPO)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

## 2. Provide and extract the pinned FoR archive

In `manual` mode, first use Colab's Files panel to upload `for_2sec_ff8c82c7.tar` and `for_2sec_REVISION.txt` into `/content/`. In `drive` mode, the files are copied from the shared raw folder. This step verifies the revision marker and extracts only the approved FoR archive. It does not access any external-test audio.

In [ ]:
import tarfile

if TRANSFER_MODE == 'drive':
    DRIVE_ROOT = Path('/content/drive/MyDrive/voice-spoofing-detection')
    ARCHIVE = DRIVE_ROOT / 'raw' / 'for_2sec_ff8c82c7.tar'
    REVISION_FILE = DRIVE_ROOT / 'raw' / 'for_2sec_REVISION.txt'
else:
    ARCHIVE = Path('/content/for_2sec_ff8c82c7.tar')
    REVISION_FILE = Path('/content/for_2sec_REVISION.txt')
    missing = [str(path) for path in (ARCHIVE, REVISION_FILE) if not path.is_file()]
    if missing:
        raise FileNotFoundError('Upload these files with the Colab Files panel: ' + ', '.join(missing))
EXPECTED_REVISION = 'ff8c82c79e7bbefef811941bac9775c9328e9055'
assert REVISION_FILE.read_text(encoding='utf-8').strip() == EXPECTED_REVISION
LOCAL_ARCHIVE = Path('/content/data/for_2sec_ff8c82c7.tar')
DATA_ROOT = Path('/content/data')
AUDIO_ROOT = DATA_ROOT / 'for_2sec'
LOCAL_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
if ARCHIVE.resolve() != LOCAL_ARCHIVE.resolve():
    shutil.copy2(ARCHIVE, LOCAL_ARCHIVE)
if AUDIO_ROOT.exists():
    shutil.rmtree(AUDIO_ROOT)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
with tarfile.open(LOCAL_ARCHIVE) as archive:
    archive.extractall(DATA_ROOT, filter='data')
print('FoR archive extracted to', AUDIO_ROOT)

## 3. Verify tests, then run the deterministic validation twice

In [ ]:
subprocess.run(['python', '-m', 'pytest', '-q'], cwd=REPO, check=True)
RUN_A = Path('/content/stage6_validation_a')
RUN_B = Path('/content/stage6_validation_b')
for output in (RUN_A, RUN_B):
    if output.exists():
        shutil.rmtree(output)
    subprocess.run([
        'python', '-m', 'src.preprocessing.validation',
        '--root', str(AUDIO_ROOT),
        '--manifest', str(REPO / 'data/manifests/for2sec_frozen_manifest.csv'),
        '--profile', str(REPO / 'configs/preprocessing.yaml'),
        '--out', str(output),
    ], cwd=REPO, check=True)

In [ ]:
import hashlib, json

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

names = ['shortcut_screen.csv', 'invariance.csv', 'profile_report.json', 'profile_report.md']
hashes_a = {name: sha256(RUN_A / name) for name in names}
hashes_b = {name: sha256(RUN_B / name) for name in names}
assert hashes_a == hashes_b
determinism = {
    'code_commit': IMPLEMENTATION_COMMIT,
    'byte_identical': True,
    'run_a_sha256': hashes_a,
    'run_b_sha256': hashes_b,
}
Path('/content/stage6_determinism.json').write_text(
    json.dumps(determinism, indent=2, sort_keys=True) + '\n', encoding='utf-8'
)
determinism

## 4. Run the disposable compute benchmark

The benchmark performs forward/backward tensor operations only. It has no optimizer, makes no weight update, writes no checkpoint, and reports no model-performance metric.

In [ ]:
BENCHMARK = Path('/content/stage6_benchmark')
if BENCHMARK.exists():
    shutil.rmtree(BENCHMARK)
subprocess.run([
    'python', '-m', 'src.preprocessing.benchmark',
    '--root', str(AUDIO_ROOT),
    '--manifest', str(REPO / 'data/manifests/for2sec_frozen_manifest.csv'),
    '--profile', str(REPO / 'configs/preprocessing.yaml'),
    '--out', str(BENCHMARK), '--device', 'cuda',
], cwd=REPO, check=True)

## 5. Export the candidate evidence

In `drive` mode, evidence is copied to the shared Stage 6 metadata folder. In `manual` mode, the notebook downloads `stage6_evidence.zip`; upload its extracted files to that same Drive folder using the normal Drive website. These are candidate F3 records. F3 becomes effective only after human review and merge.

In [ ]:
EVIDENCE = Path('/content/stage6_evidence')
if EVIDENCE.exists():
    shutil.rmtree(EVIDENCE)
EVIDENCE.mkdir()
for source in (RUN_A, BENCHMARK):
    for path in source.iterdir():
        shutil.copy2(path, EVIDENCE / path.name)
shutil.copy2('/content/stage6_determinism.json', EVIDENCE / 'determinism.json')
if TRANSFER_MODE == 'drive':
    destination = DRIVE_ROOT / 'processed' / 'metadata' / 'stage6'
    destination.mkdir(parents=True, exist_ok=True)
    for path in EVIDENCE.iterdir():
        shutil.copy2(path, destination / path.name)
    print('Candidate evidence copied to', destination)
else:
    archive = Path(shutil.make_archive('/content/stage6_evidence', 'zip', root_dir=EVIDENCE))
    from google.colab import files
    files.download(str(archive))
    print('Downloaded', archive.name, 'for manual upload to Drive processed/metadata/stage6/')
print('Do not begin Stage 7 until F3 is reviewed and merged.')